In [31]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

# =====================================
# Load Dataset
# =====================================

df = pd.read_csv(
    r"C:\A-CMSI research\Data\HMM data\HMM_Input_features.csv"
)

# =====================================
# Features
# =====================================

features = [
    "SPY_Return",
    "SPY_V",
    "RollingVol21",
    "RollingSkew21",
    "Drawdown",
    "VolOfVol",
    "VIX",
    "C_t"
]

X = df[features]

# =====================================
# Standardize Features
# =====================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# =====================================
# Train 2-State Full Covariance HMM
# =====================================

best_model = None
best_score = -np.inf

for seed in range(30):

    model = GaussianHMM(
        n_components=2,
        covariance_type="full",
        n_iter=200,
        random_state=seed
    )

    model.fit(X_scaled)

    score = model.score(X_scaled)

    if score > best_score:

        best_score = score
        best_model = model

print("="*60)
print("Best Log-Likelihood:", best_score)
print("="*60)

# =====================================
# Hidden States
# =====================================

hidden_states = best_model.predict(X_scaled)

df["State"] = hidden_states

# =====================================
# State Means
# =====================================

print("\nSTATE MEANS\n")

print(
    df.groupby("State")[[
        "SPY_Return",
        "SPY_V",
        "RollingVol21",
        "RollingSkew21",
        "Drawdown",
        "VolOfVol",
        "VIX",
        "C_t"
    ]].mean()
)

# =====================================
# Transition Matrix
# =====================================

print("\nTRANSITION MATRIX\n")

print(best_model.transmat_)

# =====================================
# Model Selection Statistics
# =====================================

logL = best_model.score(X_scaled)

n_states = best_model.n_components
n_features = X_scaled.shape[1]
n_samples = X_scaled.shape[0]

# Number of Parameters (Full Covariance)

k = (
    (n_states - 1)
    + n_states * (n_states - 1)
    + n_states * n_features
    + n_states * n_features * (n_features + 1) / 2
)

AIC = -2 * logL + 2 * k

BIC = -2 * logL + np.log(n_samples) * k

print("\nMODEL SELECTION")

print("----------------------------")
print("Number of States :", n_states)
print("Log-Likelihood   :", logL)
print("Parameters       :", int(k))
print("AIC              :", AIC)
print("BIC              :", BIC)
print("----------------------------")

Best Log-Likelihood: -17528.68888445368

STATE MEANS

       SPY_Return     SPY_V  RollingVol21  RollingSkew21  Drawdown  VolOfVol  \
State                                                                          
0        0.001021 -0.614430      0.006063      -0.150716 -0.005939  0.000917   
1        0.000049  0.484427      0.013020      -0.188035 -0.084782  0.002106   

             VIX       C_t  
State                       
0      14.827664  0.554018  
1      22.389689  0.757521  

TRANSITION MATRIX

[[0.98266255 0.01733745]
 [0.0192809  0.9807191 ]]

MODEL SELECTION
----------------------------
Number of States : 2
Log-Likelihood   : -17528.68888445368
Parameters       : 91
AIC              : 35239.37776890736
BIC              : 35769.83792982383
----------------------------
